# CXR-LT 2023 Continued Data Checks

This notebook continues the TODO list from `cxr-lt-2023.ipynb`:

1. Study-level aggregation and frontal/lateral pairing.
2. Patient timeline behavior.
3. Label co-occurrence and correlations.
4. View-conditioned prevalence.
5. Image path availability.
6. Text linkage readiness against MIMIC-CXR report metadata.

Run it from `data/00-examine-data`, matching the first notebook.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

cwd = Path.cwd()
if cwd.name != "00-examine-data":
    raise Exception("Please run this notebook from the 00-examine-data directory")

root_dir = cwd.parents[1]
data_dir = root_dir / "data"
cxr_lt_2023_dir = data_dir / "CXR-LT" / "cxr-lt-multi-label-long-tailed-classification-on-chest-x-rays-2.0.0" / "cxr-lt-2023"
mimic_cxr_dir = data_dir / "MIMIC-CXR"
mimic_cxr_jpg_dir = data_dir / "MIMIC-CXR-JPG"

print(f"root_dir: {root_dir}")
print(f"cxr_lt_2023_dir: {cxr_lt_2023_dir}")
print(f"mimic_cxr_dir exists: {mimic_cxr_dir.exists()}")
print(f"mimic_cxr_jpg_dir exists: {mimic_cxr_jpg_dir.exists()}")

## 0) Load CXR-LT Splits

This repeats the first notebook's loading conventions so the continuation can run by itself.

In [ ]:
CSV_FILES = {
    "train": "train.csv",
    "development": "development.csv",
    "test": "test.csv",
}

ID_COLUMNS = [
    "dicom_id",
    "subject_id",
    "study_id",
    "ViewPosition",
    "ViewCodeSequence_CodeMeaning",
    "path",
]


def load_dataset(filename: str) -> pd.DataFrame:
    return pd.read_csv(cxr_lt_2023_dir / filename)


def load_first_existing(paths: list[Path]) -> tuple[pd.DataFrame | None, Path | None]:
    for path in paths:
        if path.exists():
            return pd.read_csv(path), path
    return None, None


analysis_splits = {name: load_dataset(filename) for name, filename in CSV_FILES.items()}
train_df = analysis_splits["train"]
label_cols = [column for column in train_df.columns if column not in ID_COLUMNS]

split_overview_df = pd.DataFrame(
    [
        {
            "split": split_name,
            "rows": len(df),
            "subjects": df["subject_id"].nunique(),
            "studies": df["study_id"].nunique(),
            "dicoms": df["dicom_id"].nunique(),
        }
        for split_name, df in analysis_splits.items()
    ]
)

display(split_overview_df)
print(f"Detected {len(label_cols)} label columns")
label_cols

## Shared Helpers

The helpers below keep the later checks short and make missing optional MIMIC files explicit.

In [ ]:
FRONTAL_VIEWS = {"AP", "PA"}
LATERAL_VIEWS = {"LATERAL", "LL", "LPO", "RPO"}
MAJOR_VIEWS = ["AP", "PA", "LATERAL"]


def normalize_view_position(series: pd.Series) -> pd.Series:
    return series.fillna("Missing").astype(str).str.strip().str.upper().replace({"": "Missing"})


def format_view_combo(views: tuple[str, ...]) -> str:
    return " + ".join(views) if views else "Missing"


def has_any_view(views: tuple[str, ...], candidates: set[str]) -> bool:
    return bool(set(views) & candidates)


def parse_mimic_study_datetime(df: pd.DataFrame) -> pd.Series:
    if "StudyDate" not in df.columns or "StudyTime" not in df.columns:
        return pd.Series(pd.NaT, index=df.index)

    date_text = (
        df["StudyDate"]
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(8)
    )
    time_text = (
        df["StudyTime"]
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.split(".")
        .str[0]
        .str.zfill(6)
        .str[:6]
    )
    return pd.to_datetime(date_text + time_text, format="%Y%m%d%H%M%S", errors="coerce")


def build_study_view_table(df: pd.DataFrame, label_columns: list[str]) -> pd.DataFrame:
    temp = df.assign(_view=normalize_view_position(df["ViewPosition"]))
    group_keys = ["subject_id", "study_id"]

    counts = (
        temp.groupby(group_keys, as_index=False)
        .agg(
            image_count=("dicom_id", "size"),
            unique_dicoms=("dicom_id", "nunique"),
        )
    )
    views = (
        temp.groupby(group_keys)["_view"]
        .agg(lambda values: tuple(sorted(set(values))))
        .rename("views")
        .reset_index()
    )
    study_labels = (
        temp.groupby(group_keys)[label_columns]
        .max()
        .sum(axis=1)
        .rename("positive_labels_per_study")
        .reset_index()
    )

    study_table = counts.merge(views, on=group_keys).merge(study_labels, on=group_keys)
    study_table["view_combo"] = study_table["views"].map(format_view_combo)
    study_table["has_frontal"] = study_table["views"].map(lambda views: has_any_view(views, FRONTAL_VIEWS))
    study_table["has_lateral"] = study_table["views"].map(lambda views: has_any_view(views, LATERAL_VIEWS))
    study_table["has_pa"] = study_table["views"].map(lambda views: "PA" in views)
    study_table["has_ap"] = study_table["views"].map(lambda views: "AP" in views)
    study_table["has_frontal_and_lateral"] = study_table["has_frontal"] & study_table["has_lateral"]
    return study_table


def summarize_study_views(split_frames: dict[str, pd.DataFrame], label_columns: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    tables = []
    summary_rows = []

    for split_name, df in split_frames.items():
        study_table = build_study_view_table(df, label_columns).assign(split=split_name)
        tables.append(study_table)
        summary_rows.append(
            {
                "split": split_name,
                "studies": len(study_table),
                "single_image_studies_pct": (study_table["image_count"] == 1).mean() * 100,
                "median_images_per_study": study_table["image_count"].median(),
                "frontal_and_lateral_studies_pct": study_table["has_frontal_and_lateral"].mean() * 100,
                "pa_and_lateral_studies_pct": (study_table["has_pa"] & study_table["has_lateral"]).mean() * 100,
                "ap_and_lateral_studies_pct": (study_table["has_ap"] & study_table["has_lateral"]).mean() * 100,
                "median_positive_labels_per_study": study_table["positive_labels_per_study"].median(),
            }
        )

    all_studies = pd.concat(tables, ignore_index=True)
    summary = pd.DataFrame(summary_rows)
    return all_studies, summary


def plot_top_view_combos(study_table: pd.DataFrame, top_n: int = 12) -> None:
    combo_order = (
        study_table.query("split == 'train'")["view_combo"]
        .value_counts()
        .head(top_n)
        .index
    )
    plot_df = study_table[study_table["view_combo"].isin(combo_order)]

    plt.figure(figsize=(12, 5))
    sns.countplot(data=plot_df, y="view_combo", hue="split", order=combo_order)
    plt.title(f"Top {top_n} study view combinations")
    plt.xlabel("Study count")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

## 1) Study-Level Aggregation

This checks whether a `study_id` usually has a frontal view, a lateral view, or both. This matters for multi-view fusion because the image-level rows are not the same unit as a clinical exam.

In [ ]:
study_table_df, study_view_summary_df = summarize_study_views(analysis_splits, label_cols)

display(study_view_summary_df)
display(
    study_table_df
    .query("split == 'train'")
    [
        [
            "subject_id",
            "study_id",
            "image_count",
            "view_combo",
            "has_frontal_and_lateral",
            "positive_labels_per_study",
        ]
    ]
    .head(10)
)

plot_top_view_combos(study_table_df, top_n=12)

## 2) Patient Timeline Behavior

The CXR-LT CSVs do not include acquisition time, so this joins MIMIC-CXR-JPG metadata when available. MIMIC dates are shifted, but within-patient ordering and intervals are still useful for repeat-exam analysis.

In [ ]:
metadata_df, metadata_path = load_first_existing(
    [
        mimic_cxr_jpg_dir / "mimic-cxr-2.0.0-metadata.csv.gz",
        mimic_cxr_jpg_dir / "mimic-cxr-2.0.0-metadata.csv",
    ]
)

if metadata_df is None:
    print("MIMIC-CXR-JPG metadata was not found. Timeline cells will fall back to study_id ordering.")
    metadata_columns = ["dicom_id", "study_datetime"]
    metadata_for_join = pd.DataFrame(columns=metadata_columns)
else:
    print(f"Loaded metadata: {metadata_path}")
    metadata_df["study_datetime"] = parse_mimic_study_datetime(metadata_df)
    metadata_columns = [
        column
        for column in [
            "dicom_id",
            "StudyDate",
            "StudyTime",
            "study_datetime",
            "PerformedProcedureStepDescription",
            "ProcedureCodeSequence_CodeMeaning",
            "Rows",
            "Columns",
        ]
        if column in metadata_df.columns
    ]
    metadata_for_join = metadata_df[metadata_columns].copy()

all_labeled_images_df = pd.concat(
    [df.assign(split=split_name) for split_name, df in analysis_splits.items()],
    ignore_index=True,
).merge(metadata_for_join, on="dicom_id", how="left")

print(f"Rows with parsed study_datetime: {all_labeled_images_df['study_datetime'].notna().sum():,} / {len(all_labeled_images_df):,}")
display(all_labeled_images_df.head(3))

In [ ]:
def build_study_timeline(images_df: pd.DataFrame, label_columns: list[str]) -> pd.DataFrame:
    temp = images_df.assign(_view=normalize_view_position(images_df["ViewPosition"]))
    group_keys = ["split", "subject_id", "study_id"]

    base = (
        temp.groupby(group_keys, as_index=False)
        .agg(
            image_count=("dicom_id", "size"),
            study_datetime=("study_datetime", "min"),
            first_dicom_id=("dicom_id", "first"),
        )
    )
    views = (
        temp.groupby(group_keys)["_view"]
        .agg(lambda values: tuple(sorted(set(values))))
        .rename("views")
        .reset_index()
    )
    labels = (
        temp.groupby(group_keys)[label_columns]
        .max()
        .sum(axis=1)
        .rename("positive_labels_per_study")
        .reset_index()
    )

    timeline = base.merge(views, on=group_keys).merge(labels, on=group_keys)
    timeline["view_combo"] = timeline["views"].map(format_view_combo)
    timeline = timeline.sort_values(["split", "subject_id", "study_datetime", "study_id"], na_position="last")
    timeline["days_since_previous_study"] = (
        timeline.groupby(["split", "subject_id"])["study_datetime"]
        .diff()
        .dt.total_seconds()
        .div(86400)
    )
    return timeline


study_timeline_df = build_study_timeline(all_labeled_images_df, label_cols)

patient_timeline_summary_df = (
    study_timeline_df.groupby(["split", "subject_id"], as_index=False)
    .agg(
        study_count=("study_id", "nunique"),
        first_study_datetime=("study_datetime", "min"),
        last_study_datetime=("study_datetime", "max"),
        total_images=("image_count", "sum"),
    )
)
patient_timeline_summary_df["span_days"] = (
    patient_timeline_summary_df["last_study_datetime"] - patient_timeline_summary_df["first_study_datetime"]
).dt.total_seconds().div(86400)

split_patient_summary_df = (
    patient_timeline_summary_df.groupby("split")
    .agg(
        patients=("subject_id", "nunique"),
        patients_with_multiple_studies=("study_count", lambda values: int((values > 1).sum())),
        median_studies_per_patient=("study_count", "median"),
        max_studies_per_patient=("study_count", "max"),
        median_span_days=("span_days", "median"),
        max_span_days=("span_days", "max"),
    )
    .reset_index()
)
split_patient_summary_df["multiple_study_patient_pct"] = (
    split_patient_summary_df["patients_with_multiple_studies"] / split_patient_summary_df["patients"] * 100
)

display(split_patient_summary_df)
display(
    patient_timeline_summary_df
    .sort_values(["study_count", "span_days"], ascending=[False, False])
    .head(10)
)

In [ ]:
top_patient_row = (
    patient_timeline_summary_df
    .sort_values(["study_count", "span_days"], ascending=[False, False])
    .head(1)
)

if top_patient_row.empty:
    print("No patient timeline rows available.")
else:
    top_split = top_patient_row.iloc[0]["split"]
    top_subject_id = top_patient_row.iloc[0]["subject_id"]
    print(f"Example repeat-exam patient: split={top_split}, subject_id={top_subject_id}")
    display(
        study_timeline_df
        .query("split == @top_split and subject_id == @top_subject_id")
        [
            [
                "split",
                "subject_id",
                "study_id",
                "study_datetime",
                "days_since_previous_study",
                "image_count",
                "view_combo",
                "positive_labels_per_study",
            ]
        ]
    )

## 3) Label Co-occurrence And Correlations

Use the training split for these checks so downstream choices are driven by train data only. The correlation heatmap uses Pearson correlation on binary labels, which is the phi coefficient for two binary variables.

In [ ]:
def label_pair_correlation_table(df: pd.DataFrame, label_columns: list[str]) -> pd.DataFrame:
    corr = df[label_columns].corr()
    pairs = []
    for left_idx, left_label in enumerate(label_columns):
        for right_label in label_columns[left_idx + 1 :]:
            pairs.append(
                {
                    "left_label": left_label,
                    "right_label": right_label,
                    "correlation": corr.loc[left_label, right_label],
                    "cooccurrence_count": int(((df[left_label] == 1) & (df[right_label] == 1)).sum()),
                    "left_positive_count": int(df[left_label].sum()),
                    "right_positive_count": int(df[right_label].sum()),
                }
            )
    return pd.DataFrame(pairs).dropna(subset=["correlation"])


train_corr_df = train_df[label_cols].corr()
pair_corr_df = label_pair_correlation_table(train_df, label_cols)

plt.figure(figsize=(14, 12))
sns.heatmap(train_corr_df, cmap="vlag", center=0, vmin=-1, vmax=1, square=True, linewidths=0.2)
plt.title("Train label correlation matrix")
plt.tight_layout()
plt.show()

print("Most positively correlated label pairs")
display(pair_corr_df.sort_values("correlation", ascending=False).head(15))

print("Most negatively correlated label pairs")
display(pair_corr_df.sort_values("correlation", ascending=True).head(15))

In [ ]:
train_cooccurrence_df = train_df[label_cols].T.dot(train_df[label_cols])
label_order = train_df[label_cols].sum().sort_values(ascending=False).index

plt.figure(figsize=(14, 12))
sns.heatmap(
    train_cooccurrence_df.loc[label_order, label_order],
    cmap="mako",
    square=True,
    linewidths=0.2,
)
plt.title("Train label co-occurrence counts")
plt.tight_layout()
plt.show()

display(train_cooccurrence_df.loc[label_order, label_order].iloc[:10, :10])

## 4) View-Conditioned Prevalence

This compares label rates for `AP`, `PA`, and `LATERAL`. It is useful for checking whether a label is view-dependent enough that frontal and lateral encoders should be treated separately.

In [ ]:
def view_conditioned_prevalence(
    split_frames: dict[str, pd.DataFrame],
    label_columns: list[str],
    major_views: list[str] = MAJOR_VIEWS,
) -> pd.DataFrame:
    rows = []
    for split_name, df in split_frames.items():
        temp = df.assign(_view=normalize_view_position(df["ViewPosition"]))
        for view_name, view_df in temp[temp["_view"].isin(major_views)].groupby("_view"):
            for label in label_columns:
                positives = int(view_df[label].sum())
                rows.append(
                    {
                        "split": split_name,
                        "ViewPosition": view_name,
                        "label": label,
                        "row_count": len(view_df),
                        "positive_count": positives,
                        "positive_rate_pct": positives / len(view_df) * 100 if len(view_df) else np.nan,
                    }
                )
    return pd.DataFrame(rows)


view_prevalence_df = view_conditioned_prevalence(analysis_splits, label_cols)
view_counts_df = (
    view_prevalence_df[["split", "ViewPosition", "row_count"]]
    .drop_duplicates()
    .sort_values(["split", "ViewPosition"])
)

display(view_counts_df)

top_train_labels = train_df[label_cols].sum().sort_values(ascending=False).head(12).index.tolist()
plot_df = view_prevalence_df.query("split == 'train' and label in @top_train_labels")

plt.figure(figsize=(11, 7))
sns.barplot(data=plot_df, x="positive_rate_pct", y="label", hue="ViewPosition", order=top_train_labels)
plt.title("Train label prevalence by view position")
plt.xlabel("Positive rate (%)")
plt.ylabel("")
plt.tight_layout()
plt.show()

display(
    plot_df.pivot(index="label", columns="ViewPosition", values="positive_rate_pct")
    .loc[top_train_labels]
    .round(3)
)

## 5) Image Availability And Path Sanity

CXR-LT `path` values are relative to the MIMIC-CXR-JPG root. This checks whether the expected JPG files are present locally before dataloader work.

In [ ]:
def resolve_relative_path(root: Path, path_value: str) -> Path:
    path = Path(path_value)
    return path if path.is_absolute() else root / path


def summarize_image_paths(split_frames: dict[str, pd.DataFrame], image_root: Path) -> pd.DataFrame:
    rows = []
    for split_name, df in split_frames.items():
        paths = df["path"].dropna().astype(str)
        exists_flags = paths.map(lambda value: resolve_relative_path(image_root, value).is_file())
        rows.append(
            {
                "split": split_name,
                "rows": len(df),
                "path_values": len(paths),
                "unique_paths": paths.nunique(),
                "duplicate_path_rows": int(paths.duplicated().sum()),
                "jpg_extension_rows": int(paths.str.lower().str.endswith(".jpg").sum()),
                "existing_files": int(exists_flags.sum()),
                "missing_files": int((~exists_flags).sum()),
                "existing_file_pct": exists_flags.mean() * 100 if len(exists_flags) else np.nan,
            }
        )
    return pd.DataFrame(rows)


image_root = mimic_cxr_jpg_dir
print(f"image_root: {image_root}")
print(f"image_root exists: {image_root.exists()}")
print(f"image files directory exists: {(image_root / 'files').exists()}")

image_path_summary_df = summarize_image_paths(analysis_splits, image_root)
display(image_path_summary_df)

missing_train_paths = train_df.loc[
    ~train_df["path"].astype(str).map(lambda value: resolve_relative_path(image_root, value).is_file()),
    ["dicom_id", "subject_id", "study_id", "ViewPosition", "path"],
].head(10)

print("Sample missing train image paths")
display(missing_train_paths)

## 6) Text Linkage Readiness

This joins CXR-LT studies to the MIMIC-CXR study list. A successful index join means the label row can be paired to a report path. File existence is a separate check because this workstation may have metadata without the report text tree.

In [ ]:
study_list_df, study_list_path = load_first_existing(
    [
        mimic_cxr_dir / "cxr-study-list.csv.gz",
        mimic_cxr_dir / "cxr-study-list.csv",
    ]
)

if study_list_df is None:
    print("MIMIC-CXR study list was not found. Report linkage cannot be checked.")
    report_link_summary_df = pd.DataFrame()
    linked_reports_df = pd.DataFrame()
else:
    print(f"Loaded study list: {study_list_path}")
    study_list_for_join = study_list_df.rename(columns={"path": "report_path"})
    unique_studies_df = pd.concat(
        [
            df[["subject_id", "study_id"]].drop_duplicates().assign(split=split_name)
            for split_name, df in analysis_splits.items()
        ],
        ignore_index=True,
    )
    linked_reports_df = unique_studies_df.merge(
        study_list_for_join[["subject_id", "study_id", "report_path"]],
        on=["subject_id", "study_id"],
        how="left",
    )
    linked_reports_df["has_report_index"] = linked_reports_df["report_path"].notna()
    linked_reports_df["report_file_exists"] = linked_reports_df["report_path"].fillna("").map(
        lambda value: (mimic_cxr_dir / value).is_file() if value else False
    )

    report_link_summary_df = (
        linked_reports_df.groupby("split")
        .agg(
            studies=("study_id", "nunique"),
            studies_with_report_index=("has_report_index", "sum"),
            studies_with_report_file=("report_file_exists", "sum"),
        )
        .reset_index()
    )
    report_link_summary_df["report_index_pct"] = (
        report_link_summary_df["studies_with_report_index"] / report_link_summary_df["studies"] * 100
    )
    report_link_summary_df["report_file_pct"] = (
        report_link_summary_df["studies_with_report_file"] / report_link_summary_df["studies"] * 100
    )

    print(f"report files directory exists: {(mimic_cxr_dir / 'files').exists()}")
    display(report_link_summary_df)
    display(linked_reports_df.head(10))

In [ ]:
def read_report_text(report_path: str, max_chars: int = 3000) -> str | None:
    if not report_path:
        return None
    full_path = mimic_cxr_dir / report_path
    if not full_path.is_file():
        return None
    return full_path.read_text(errors="replace")[:max_chars]


if linked_reports_df.empty or not linked_reports_df["report_file_exists"].any():
    print("No local report text files were found to preview.")
else:
    sample_report_row = linked_reports_df.query("report_file_exists").iloc[0]
    print(
        "Sample report: "
        f"split={sample_report_row['split']}, "
        f"subject_id={sample_report_row['subject_id']}, "
        f"study_id={sample_report_row['study_id']}"
    )
    print(read_report_text(sample_report_row["report_path"]))

## Notes For The Next Modeling Pass

Use these outputs to decide whether the modular dataloader should preserve study-level grouping, whether paired frontal/lateral samples are common enough to require special batching, and whether the local machine has the image/report files required for multimodal experiments.